# 06 · Power BI model exports and final findings

Actual Superstore sample data, not simulated outputs. Run cells from top to bottom. Dollar amounts are USD.


In [1]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/project.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the extracted project folder.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.project import load, summarize, save_table, chart
pd.set_option('display.max_columns',30)
pd.set_option('display.width',160)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize':(10,5),'axes.spines.top':False,'axes.spines.right':False})

## Star schema
FactSales keeps the original line grain. DimProduct uses a generated key for each unique product ID/name/category/sub-category combination. DimCustomer contains stable customer attributes; geography stays in the fact because customers may order to multiple locations. DimDate links to Order Date. All relationships should be single direction from dimension to fact.

In [2]:
df=load()
out=ROOT/'powerbi/data';out.mkdir(exist_ok=True)
product_cols=['Product ID','Product Name','Category','Sub-Category']
products=df[product_cols].drop_duplicates().sort_values(product_cols).reset_index(drop=True)
products.insert(0,'ProductKey',np.arange(1,len(products)+1))
assert df.groupby('Customer ID')[['Customer Name','Segment']].nunique().le(1).all().all()
customers=df[['Customer ID','Customer Name','Segment']].drop_duplicates()
assert customers['Customer ID'].is_unique
fact=df.merge(products,on=product_cols,how='left',validate='many_to_one')
assert len(fact)==len(df) and fact.ProductKey.notna().all()
fact=fact.drop(columns=product_cols+['Customer Name','Segment'])
dates=pd.DataFrame({'Date':pd.date_range(f'{df.Year.min()}-01-01',f'{df.Year.max()}-12-31')})
dates['Year']=dates.Date.dt.year;dates['Month Number']=dates.Date.dt.month


dates['Month']=dates.Date.dt.month_name();dates['Year Month']=dates.Date.dt.strftime('%Y-%m')
for name,frame in [('FactSales',fact),('DimProduct',products),('DimCustomer',customers),('DimDate',dates)]:
    frame.to_csv(out/f'{name}.csv',index=False,date_format='%Y-%m-%d')
    print(name,len(frame))
assert np.isclose(fact.Sales.sum(),df.Sales.sum())
assert np.isclose(fact.Profit.sum(),df.Profit.sum())
print('Exports preserve row counts, sales and profit.')

FactSales 9994
DimProduct 1894
DimCustomer 793
DimDate 1461
Exports preserve row counts, sales and profit.


In [3]:
category=summarize(df,'Category');states=summarize(df,'State').sort_values('Profit')
sub=summarize(df,'Sub-Category')
annual=df.groupby('Year').Sales.sum()
report=f"""# Findings from the executed sample dataset

- Lines: {len(df):,}; orders: {df['Order ID'].nunique():,}; customers: {df['Customer ID'].nunique():,}.
- Sales: ${df.Sales.sum():,.2f}; profit: ${df.Profit.sum():,.2f}; weighted margin: {100*df.Profit.sum()/df.Sales.sum():.2f}%.
- Highest-sales category: {category.index[0]} (${category.iloc[0].Sales:,.2f}).
- Lowest-profit state: {states.index[0]} (${states.iloc[0].Profit:,.2f}).
- Tables profit: ${sub.loc['Tables','Profit']:,.2f}; Bookcases profit: ${sub.loc['Bookcases','Profit']:,.2f}.

## Actions and validation
1. Audit discount and product mix for loss-making furniture lines; pilot changes before rollout.
2. Review the lowest-profit states at sub-category level; avoid assuming geography causes losses.
3. Use monthly patterns for planning, then validate with recent demand and inventory data.
4. Review high-value customer segments using profitability as well as sales.

All values are computed from historical sample data. Recommendations are hypotheses, not realized business outcomes. No dashboard has been built or validated in Power BI Desktop in this package.
"""
(ROOT/'reports/FINDINGS.md').write_text(report,encoding='utf-8')
print(report)

# Findings from the executed sample dataset

- Lines: 9,994; orders: 5,009; customers: 793.
- Sales: $2,297,200.86; profit: $286,397.02; weighted margin: 12.47%.
- Highest-sales category: Technology ($836,154.03).
- Lowest-profit state: Texas ($-25,729.36).
- Tables profit: $-17,725.48; Bookcases profit: $-3,472.56.

## Actions and validation
1. Audit discount and product mix for loss-making furniture lines; pilot changes before rollout.
2. Review the lowest-profit states at sub-category level; avoid assuming geography causes losses.
3. Use monthly patterns for planning, then validate with recent demand and inventory data.
4. Review high-value customer segments using profitability as well as sales.

All values are computed from historical sample data. Recommendations are hypotheses, not realized business outcomes. No dashboard has been built or validated in Power BI Desktop in this package.

